Reducing Dataset - removing duplicates, variance and corr reduction

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold


# ============================================================
# CONFIG
# ============================================================

INPUT_FILE = r"C:\1.Revanth\Projects\research\pipeline_cache\ampds_behavior_context_labeled_features.csv"
OUTPUT_FILE = r"C:\1.Revanth\Projects\research\pipeline_cache\ampds_behavior_context_reduced_features.csv"

# Variance threshold
VARIANCE_THRESHOLD = 1e-5

# Correlation threshold
CORRELATION_THRESHOLD = 0.95


# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv(INPUT_FILE)

# Keep window_id separately
window_id_col = None

if 'window_id' in df.columns:
    window_id_col = df['window_id'].copy()
    df = df.drop(columns=['window_id'])

# ============================================================
# KEEP LABEL COLUMNS SEPARATE
# ============================================================

label_cols = [
    "is_anomaly",
    "anomaly_type"
]

X = df.drop(columns=label_cols).copy()

print("Original feature count:", X.shape[1])


# ============================================================
# 1. REMOVE EXACT DUPLICATE FEATURES
# ============================================================

duplicate_columns = X.T.duplicated()

duplicate_names = X.columns[
    duplicate_columns
].tolist()

X = X.loc[
    :,
    ~duplicate_columns
]

print(
    "Removed duplicate features:",
    len(duplicate_names)
)

print(
    "After duplicate removal:",
    X.shape[1]
)


# ============================================================
# 2. REMOVE LOW-VARIANCE FEATURES
# ============================================================

selector = VarianceThreshold(
    threshold=VARIANCE_THRESHOLD
)

X_variance = selector.fit_transform(X)

kept_columns = X.columns[
    selector.get_support()
]

removed_variance = X.columns[
    ~selector.get_support()
].tolist()

X = pd.DataFrame(
    X_variance,
    columns=kept_columns,
    index=X.index
)

print(
    "Removed low-variance features:",
    len(removed_variance)
)

print(
    "After variance removal:",
    X.shape[1]
)


# ============================================================
# 3. REMOVE HIGHLY CORRELATED FEATURES
# ============================================================

corr_matrix = X.corr().abs()

upper_triangle = corr_matrix.where(
    np.triu(
        np.ones(
            corr_matrix.shape
        ),
        k=1
    ).astype(bool)
)

correlated_features = [
    column
    for column in upper_triangle.columns
    if any(
        upper_triangle[column]
        > CORRELATION_THRESHOLD
    )
]

X = X.drop(
    columns=correlated_features
)

print(
    "Removed correlated features:",
    len(correlated_features)
)

print(
    "Final feature count:",
    X.shape[1]
)


# ============================================================
# 4. ADD LABELS BACK
# ============================================================

df_reduced = X.copy()

df_reduced["is_anomaly"] = df[
    "is_anomaly"
].values

df_reduced["anomaly_type"] = df[
    "anomaly_type"
].values

if window_id_col is not None:
    df_reduced.insert(
        0,
        "window_id",
        window_id_col.values
    )


# ============================================================
# 5. SAVE
# ============================================================

df_reduced.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n===================================")
print("DONE")
print("===================================")

print(
    "Original features:",
    df.shape[1] - len(label_cols)
)

print(
    "Final features:",
    X.shape[1]
)

reduction_pct = 100 * (1 - X.shape[1] / (df.shape[1] - len(label_cols)))
print("Reduction:", f"{reduction_pct:.2f}%")

print(
    "Saved:",
    OUTPUT_FILE
)

Reducing Dataset from FeatureWiz